In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

In [3]:
import sys
import os
sys.path.append(os.path.abspath('../..'))
from dl2_reports import DL2Report

In [4]:
df = pd.read_csv("penguins_size.csv")

df = df.dropna()
df.head()

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,MALE


In [5]:
# Test dataset with datetime columns (tz-aware + naive)
import pandas as pd
import numpy as np

# tz-aware timestamps
when_utc = pd.date_range("2025-01-01 00:00:00", periods=5, freq="h", tz="UTC")
# naive timestamps (will be treated as UTC by DL2Report.add_df)
when_naive = pd.date_range("2025-01-01 00:00:00", periods=5, freq="h")

_df_dt = pd.DataFrame(
    {
        "when_utc": when_utc,
        "when_naive": when_naive,
        "value": [10, 20, 30, 40, 50],
        "nan_col": [np.nan, np.nan, np.nan, np.nan, np.nan]
    }
)

_dt_report = DL2Report(title="Datetime smoke test")
_dt_report.add_df("dt_iso", _df_dt.copy(), format="records", compress=False, timestamp_format="iso")
_dt_report.add_df("dt_epoch", _df_dt.copy(), format="records", compress=False, timestamp_format="epoch")

#print("ISO sample:", _dt_report.datasets["dt_iso"]["data"][0])
#print("Epoch sample:", _dt_report.datasets["dt_epoch"]["data"][0])

# show the DataFrame too
_df_dt

,when_utc,when_naive,value,nan_col
0,2025-01-01 00:00:00+00:00,2025-01-01 00:00:00,10,NaN
1,2025-01-01 01:00:00+00:00,2025-01-01 01:00:00,20,NaN
2,2025-01-01 02:00:00+00:00,2025-01-01 02:00:00,30,NaN
3,2025-01-01 03:00:00+00:00,2025-01-01 03:00:00,40,NaN
4,2025-01-01 04:00:00+00:00,2025-01-01 04:00:00,50,NaN


In [6]:
# Render the datetime smoke-test as its own mini report (tables)
# Keeps it separate from the penguins report.

_dt_page = _dt_report.add_page("Datetime")

_dt_row = _dt_page.add_row()
_dt_row.add_table(dataset_id="dt_iso", title="Datetime (ISO)", pageSize=10)

_dt_row = _dt_page.add_row()
_dt_row.add_table(dataset_id="dt_epoch", title="Datetime (Epoch seconds)", pageSize=10)

_dt_report.save("datetime_smoke.html")
_dt_report.show()

_dt_report

In [7]:
report = DL2Report(
    title="Pengins", 
    description="A report about penguin size. Based on data from https://www.kaggle.com/datasets/amulyas/penguin-size-dataset", 
    author="Kameron Brooks",
    compress_visuals=False,
    cdn_url="https://d12owwy7533nor.cloudfront.net/dist/"
    )

report.add_df(
    "penguins", 
    df, 
    format="table", 
    compress=True
)

# Create synthetic data for Area Chart demo
# Using simple integers/lists to ensure compatibility
growth_df = pd.DataFrame({
    "Month_Num": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    "Gentoo_Population": [100, 115, 125, 140, 145, 160, 175, 180, 195, 210, 225, 240],
    "Adelie_Population": [80, 85, 95, 100, 110, 115, 125, 130, 135, 145, 150, 155]
})
report.add_df("growth", growth_df, format="table", compress=True)

page = report.add_page("Main")

row = page.add_row()

vis1 = row.add_scatter(
    dataset_id="penguins", 
    x_column="flipper_length_mm", 
    y_column="body_mass_g", 
    xAxisLabel="Flipper Length", 
    yAxisLabel="Body Mass",
    title="Flipper vs Body Mass"
)
vis1.add_trend(coefficients=4, line_style="dashed", color="#FF5733")


vis2 = row.add_scatter(
    dataset_id="penguins", 
    x_column="culmen_length_mm", 
    y_column="body_mass_g", 
    xAxisLabel="Culmen Length", 
    yAxisLabel="Body Mass",
    title="Culmen vs Body Mass",
    showTrendline=True
)

# Add Area Chart to show population growth
row = page.add_row()
row.add_area(
    dataset_id="growth",
    x_column="Month_Num",
    y_columns=["Gentoo_Population", "Adelie_Population"],
    title="Estimated Population Growth (Area)",
    smooth=True,
    fill_opacity=0.3,
    show_markers=True,
    y_axis_label="Population",
    x_axis_label="Month"
)

row = page.add_row()
row.add_table(dataset_id="penguins", pageSize=20)

report.save("pengins2.html")
report.show()

c:\Users\kameron\Documents\Projects\Web Projects\superbabycart\datalys2-reporting-python-api\dl2_reports\report.py:126: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(sample)
c:\Users\kameron\Documents\Projects\Web Projects\superbabycart\datalys2-reporting-python-api\dl2_reports\report.py:126: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(sample)
c:\Users\kameron\Documents\Projects\Web Projects\superbabycart\datalys2-reporting-python-api\dl2_reports\report.py:126: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(sample)


In [8]:
report